In [25]:
def write_priors_table_tabular_math_single(source_path, out_path):
    """
    One LaTeX table (plain tabular) with two in-table headers:
      - Global (universal) NumPyro priors
      - Object-level NumPyro priors (inside numpyro.plate)

    Columns: Parameter | Distribution
    Distributions: N, U, LN, and truncated N_[a,b]
    """
    import ast, datetime, re
    from pathlib import Path

    src_path = Path(source_path)
    src = src_path.read_text(encoding="utf-8")
    tree = ast.parse(src)

    # ---------- AST helpers ----------
    def is_numpyro_sample(call):
        f = call.func
        return (isinstance(f, ast.Attribute) and isinstance(f.value, ast.Name)
                and f.value.id == "numpyro" and f.attr == "sample") \
            or (isinstance(f, ast.Name) and f.id == "sample")

    def is_numpyro_plate(node):
        if not isinstance(node, ast.Call): return False
        f = node.func
        return (isinstance(f, ast.Attribute) and isinstance(f.value, ast.Name)
                and f.value.id == "numpyro" and f.attr == "plate") \
            or (isinstance(f, ast.Name) and f.id == "plate")

    def dist_name_args(node):
        if not isinstance(node, ast.Call):
            frag = ast.get_source_segment(src, node)
            return (frag or "<expr>"), ""
        f = node.func
        dname = f.attr if isinstance(f, ast.Attribute) else (f.id if isinstance(f, ast.Name) else (ast.get_source_segment(src, f) or "<expr>"))
        parts = []
        for a in node.args:
            parts.append(ast.get_source_segment(src, a) or "<arg>")
        for kw in node.keywords:
            key = kw.arg or ""
            val = ast.get_source_segment(src, kw.value) or "<val>"
            parts.append(f"{key}={val}" if key else val)
        return dname, ", ".join(parts)

    class Visitor(ast.NodeVisitor):
        def __init__(self): self.rows = []; self.stack = []
        def visit_With(self, node):
            pushed = 0
            for it in node.items:
                ctx = it.context_expr
                if isinstance(ctx, ast.Call) and is_numpyro_plate(ctx):
                    label = (ctx.args and isinstance(ctx.args[0], ast.Constant) and isinstance(ctx.args[0].value, str) and ctx.args[0].value) or "plate"
                    self.stack.append(label); pushed += 1
            for stmt in node.body: self.visit(stmt)
            for _ in range(pushed): self.stack.pop()
        visit_AsyncWith = visit_With
        def visit_Call(self, node):
            if is_numpyro_sample(node):
                if node.args:
                    a0 = node.args[0]
                    name = a0.value if isinstance(a0, ast.Constant) and isinstance(a0.value, str) \
                           else (ast.get_source_segment(src, a0) or "<name>")
                else:
                    name = "<name>"
                dname, dargs = ("<unknown>", "")
                if len(node.args) >= 2:
                    dname, dargs = dist_name_args(node.args[1])
                scope = "Global" if not self.stack else "Object-level"
                self.rows.append({"Parameter": name, "Distribution": dname, "Arguments": dargs, "Scope": scope})
            self.generic_visit(node)

    v = Visitor(); v.visit(tree)
    rows = v.rows
    globals_rows = [r for r in rows if r["Scope"] == "Global"]
    object_rows  = [r for r in rows if r["Scope"] != "Global"]

    # ---------- LaTeX formatting (SAFE) ----------
    GREEK = {
        "alpha": r"\alpha","beta": r"\beta","gamma": r"\gamma","delta": r"\delta",
        "epsilon": r"\epsilon","zeta": r"\zeta","eta": r"\eta","theta": r"\theta",
        "iota": r"\iota","kappa": r"\kappa","lambda": r"\lambda","lam": r"\lambda",
        "mu": r"\mu","nu": r"\nu","xi": r"\xi","pi": r"\pi","rho": r"\rho",
        "sigma": r"\sigma","tau": r"\tau","upsilon": r"\upsilon","phi": r"\phi",
        "chi": r"\chi","psi": r"\psi","omega": r"\omega",
    }
    NUM = re.compile(r"(\d+(?:\.\d*)?|\.\d+)$")
    NP_LIKE = re.compile(r"\b(?:jnp|np)\.")
    IDENT_SAFE = re.compile(r'(?<!\\)\b([A-Za-z][A-Za-z0-9_]*?)\b')  # skip TeX commands

    def tok_math(tok: str) -> str:
        if tok in GREEK: return GREEK[tok]
        if tok == "log10": return r"\log_{10}"
        if tok == "log":   return r"\log"
        if NUM.fullmatch(tok): return tok
        return r"\mathrm{" + tok + "}"

    def ident_to_math(name: str) -> str:
        parts = name.split("_")
        # log_* → \log <rest>
        if parts[0] == "log" and len(parts) > 1:
            head_sym = tok_math(parts[1])
            if len(parts) == 2:
                return f"$\\log {head_sym}$"
            subparts = []
            for p in parts[2:]:
                m = re.match(r"([A-Za-z]+)(\d+)$", p)
                if m:
                    subparts.append(tok_math(m.group(1)) + "_{" + m.group(2) + "}")
                else:
                    subparts.append(tok_math(p))
            subs = ",".join(subparts)
            return f"$\\log {head_sym}_{{{subs}}}$"
        # general
        head = tok_math(parts[0])
        if len(parts) == 1:
            return f"${head}$"
        subparts = []
        for p in parts[1:]:
            m = re.match(r"([A-Za-z]+)(\d+)$", p)
            if m:
                subparts.append(tok_math(m.group(1)) + "_{" + m.group(2) + "}")
            else:
                subparts.append(tok_math(p))
        subs = ",".join(subparts)
        return f"${head}_{{{subs}}}$"

    def ident_to_math_naked(name: str) -> str:
        parts = name.split("_")
        if parts[0] == "log" and len(parts) > 1:
            head_sym = tok_math(parts[1])
            if len(parts) == 2:
                return r"\log " + head_sym
            subparts = []
            for p in parts[2:]:
                m = re.match(r"([A-Za-z]+)(\d+)$", p)
                if m:
                    subparts.append(tok_math(m.group(1)) + "_{" + m.group(2) + "}")
                else:
                    subparts.append(tok_math(p))
            return r"\log " + head_sym + "_{" + ",".join(subparts) + "}"
        head = tok_math(parts[0])
        if len(parts) == 1:
            return head
        subparts = []
        for p in parts[1:]:
            m = re.match(r"([A-Za-z]+)(\d+)$", p)
            if m:
                subparts.append(tok_math(m.group(1)) + "_{" + m.group(2) + "}")
            else:
                subparts.append(tok_math(p))
        return head + "_{" + ",".join(subparts) + "}"

    def mathify_expr(expr: str) -> str:
        """TeX for a Python-ish expression; returns TeX without surrounding $."""
        s = NP_LIKE.sub("", expr or "")
        # Use function replacements so backslashes aren't treated as regex escapes
        s = re.sub(r"\blog10\s*\(", lambda m: r"\log_{10}(", s)
        s = re.sub(r"\blog\s*\(",   lambda m: r"\log(", s)

        # --- handle '*' smartly ---
        # number * \log(...)  ->  number\log(...)
        def repl_num_log(m):
            left, right = m.group(1), m.group(2)  # right starts with '\log'
            return left + right
        s = re.sub(r"(\d+(?:\.\d+)?)\s*\*\s*(\\log)", repl_num_log, s)

        # var * var  ->  var\cdot var   (must use lambda to avoid '\c' escape error)
        s = re.sub(r"(\w+)\s*\*\s*(\w+)", lambda m: m.group(1) + r"\cdot " + m.group(2), s)

        # identifiers to math (don’t touch things already starting with '\')
        s = IDENT_SAFE.sub(lambda m: ident_to_math_naked(m.group(1)), s)
        return s



    def bounds_tex(low, high):
        lo = mathify_expr(low) if low else r"-\infty"
        hi = mathify_expr(high) if high else r"\infty"
        return f"[{lo},{hi}]"

    def dist_to_fancy(dname: str, args: str) -> str:
        dn = dname.lower()
        # keyword bounds
        low_m  = re.search(r"\blow\s*=\s*([^,)\s]+)",  args)
        high_m = re.search(r"\bhigh\s*=\s*([^,)\s]+)", args)
        low  = low_m.group(1)  if low_m  else None
        high = high_m.group(1) if high_m else None
        # positional (simple split)
        pos = [a.strip() for a in args.split(",") if "=" not in a]
        mu = pos[0] if len(pos) >= 1 else None
        sd = pos[1] if len(pos) >= 2 else None

        if "truncatednormal" in dn:
            mu_tex = mathify_expr(mu or r"\cdot")
            sd_tex = mathify_expr(sd or r"\cdot")
            bnd = bounds_tex(low, high)
            return rf"$\mathcal{{N}}_{{{bnd}}}({mu_tex},{sd_tex})$"
        if "normal" in dn and "log" not in dn:
            mu_tex = mathify_expr(mu or r"\cdot")
            sd_tex = mathify_expr(sd or r"\cdot")
            return rf"$\mathcal{{N}}({mu_tex},{sd_tex})$"
        if "uniform" in dn:
            a = pos[0] if len(pos) >= 1 else r"\cdot"
            b = pos[1] if len(pos) >= 2 else r"\cdot"
            a_tex = mathify_expr(a); b_tex = mathify_expr(b)
            return rf"$\mathcal{{U}}({a_tex},{b_tex})$"
        if "lognormal" in dn or (dn == "lognormal"):
            mu_tex = mathify_expr(mu or r"\cdot")
            sd_tex = mathify_expr(sd or r"\cdot")
            return rf"$\mathcal{{LN}}({mu_tex},{sd_tex})$"
        return r"$\mathrm{" + dname + r"}$"

    # ---------- Assemble single table ----------
    def row_to_tex(r):
        param = ident_to_math(r["Parameter"])                  # with $...$
        dist  = dist_to_fancy(r["Distribution"], r["Arguments"])  # already $...$
        return f"{param} & {dist} \\\\"

    # Sort by first appearance within each section
    def sect(rows, title):
        if not rows:
            return [r"\multicolumn{2}{l}{\textbf{" + title + r"}} \\",
                    r"\multicolumn{2}{c}{\emph{None found}} \\"]
        out = [r"\multicolumn{2}{l}{\textbf{" + title + r"}} \\"]
        out.append(r"\addlinespace[0.25em]")
        out += [row_to_tex(r) for r in rows]
        out.append(r"\addlinespace[0.5em]")
        return out

    now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    preamble = "% Auto-generated on " + now + "\n"

    lines = []
    lines.append("\\begin{table}[!ht]")
    lines.append("\\centering")
    lines.append("\\small")
    lines.append("\\begin{tabular}{l l}")
    lines.append("\\toprule")
    lines.append("Parameter & Distribution \\\\")
    lines.append("\\midrule")
    lines += sect(globals_rows, "Universal priors")
    lines += sect(object_rows,  "Object-level priors")
    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    lines.append("\\caption{NumPyro priors by scope, parsed automatically from \\texttt{" + src_path.name.replace("_", r"\_") + "}.}")
    lines.append("\\end{table}")

    latex = preamble + "\n".join(lines) + "\n"
    Path(out_path).write_text(latex, encoding="utf-8")
    return latex


write_priors_table_tabular_math_single("multiband_fit.py", out_path="plots/hubble/priors_table.tex")

'% Auto-generated on 2025-09-22 01:46\n\\begin{table}[!ht]\n\\centering\n\\small\n\\begin{tabular}{l l}\n\\toprule\nParameter & Distribution \\\\\n\\midrule\n\\multicolumn{2}{l}{\\textbf{Universal priors}} \\\\\n\\addlinespace[0.25em]\n$\\eta_{\\mathrm{A}_{1},\\mathrm{mean}}$ & $\\mathcal{U}(-5.0,0.0)$ \\\\\n$\\eta_{\\tau_{1},\\mathrm{mean}}$ & $\\mathcal{U}(-1.0,5.0)$ \\\\\n$\\eta_{\\mathrm{break}}$ & $\\mathcal{LN}(\\mu,\\sigma)$ \\\\\n$\\lambda_{\\mathrm{s}}$ & $\\mathcal{N}(2500.0,100.0)$ \\\\\n$\\log \\sigma_{\\eta,\\mathrm{A}_{1}}$ & $\\mathcal{N}(\\log(0.1),0.2)$ \\\\\n$\\log \\sigma_{\\eta,\\mathrm{A}_{2}}$ & $\\mathcal{N}(\\log(0.1),0.2)$ \\\\\n$\\log \\sigma_{\\eta,\\tau_{1}}$ & $\\mathcal{N}(\\log(0.1),0.2)$ \\\\\n$\\log \\sigma_{\\eta,\\tau_{2}}$ & $\\mathcal{N}(\\log(0.1),0.2)$ \\\\\n\\addlinespace[0.5em]\n\\multicolumn{2}{l}{\\textbf{Object-level priors}} \\\\\n\\addlinespace[0.25em]\n$\\eta_{\\mathrm{A}_{1}}$ & $\\mathcal{N}(\\eta_{\\mathrm{A}_{1},\\mathrm{mean}},\\sigma